# 📘 Notebook 10: Supervised Learning II: Classification

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module C: Machine Learning · Notebook 3 of 4 in this module · (10 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Distinguish classification from regression and frame problems correctly
- Understand **logistic regression** and the **sigmoid** function
- Apply **k-Nearest Neighbours** and **decision trees**
- Evaluate classifiers with a **confusion matrix**, precision, recall, and F1
- Interpret an **ROC curve** and the AUC score

> **Prerequisites:** Notebooks 08-09.
>
> 🔭 **Why classification matters here:** the medical-AI work that motivates much of modern deep learning, for example classifying a brain MRI as showing Alzheimer's disease or not, is a classification problem. The evaluation tools in this notebook (especially recall and ROC) are exactly those used to judge such models responsibly.

> ℹ️ **Setup note.** If needed: `import piplite; await piplite.install(['scikit-learn','numpy','matplotlib'])`

## 1. From predicting numbers to predicting categories

### The shift
In regression the target was a continuous number. In **classification** the target is a **category** (a *class*): spam / not spam, cat / dog / bird, disease / healthy. The model outputs a class label, usually via an underlying probability.

We will create a simple two-class (binary) dataset to work with throughout:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, class_sep=1.2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", alpha=0.6, edgecolor="k")
plt.title("Two-class classification data")
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.show()

## 2. Logistic regression

### Intuition first
Despite the name, logistic regression is a **classification** algorithm. It computes a linear score (like regression) and then squashes it into a **probability** between 0 and 1 using the **sigmoid** function. If the probability exceeds 0.5, it predicts class 1; otherwise class 0.

### The sigmoid function
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

It takes any real number and maps it smoothly into $(0, 1)$, perfect for representing a probability. This same function will reappear as an **activation function** in neural networks, so let us look at its shape:

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 4))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.title("The sigmoid function")
plt.xlabel("z (linear score)"); plt.ylabel("probability")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train, y_train)            # same fit/predict API as before
y_pred = logreg.predict(X_test)

from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

## 3. Two more intuitive classifiers

### k-Nearest Neighbours (k-NN)
The idea could not be simpler: to classify a new point, look at its **k closest** neighbours in the training data and take a majority vote. ‘You are like the company you keep.’ It does no real ‘training’, it just memorises the data and compares distances at prediction time.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print(f"k-NN accuracy: {knn.score(X_test, y_test):.3f}")

### Decision trees
A **decision tree** classifies by asking a sequence of yes/no questions about the features (‘is feature 1 > 0.5?’), splitting the data at each step, like a flowchart. They are highly interpretable, you can read the rules, but a single deep tree easily **overfits**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# max_depth limits complexity to reduce overfitting
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(f"Decision tree accuracy: {tree.score(X_test, y_test):.3f}")

## 4. Visualising decision boundaries

A classifier carves the feature space into regions, one per class. The dividing line is the **decision boundary**. Seeing it makes the differences between algorithms concrete:

In [ ]:
def plot_boundary(model, X, y, title, ax):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap="bwr", alpha=0.3)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k", s=20)
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_boundary(logreg, X_test, y_test, "Logistic Regression", axes[0])
plot_boundary(knn,    X_test, y_test, "k-NN (k=5)", axes[1])
plot_boundary(tree,   X_test, y_test, "Decision Tree", axes[2])
plt.tight_layout()
plt.show()

Notice the **shapes** of the boundaries: logistic regression draws a straight line, k-NN produces a wavy boundary that follows local data, and the decision tree creates rectangular regions (because each split is a threshold on one feature).

## 5. Evaluating classifiers properly: the confusion matrix

### Why accuracy alone is not enough
Recall from Notebook 08 that accuracy can hide serious failures. The **confusion matrix** breaks predictions into four cells and is the foundation of every richer metric. For a binary problem (positive / negative):

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

From these four numbers:
- **Precision** $= \dfrac{TP}{TP + FP}$, when we predict positive, how often are we right?
- **Recall** $= \dfrac{TP}{TP + FN}$, of all real positives, how many did we catch?
- **F1** $= 2 \cdot \dfrac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$, their balanced average.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = logreg.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)
print()
print(classification_report(y_test, y_pred, digits=3))

> 🧠 **In high-stakes domains, recall and precision pull in different directions.** In medical screening, a **false negative** (missing a real disease) is usually far worse than a **false positive** (a false alarm). There we deliberately favour high recall, accepting more false positives. The ‘right’ balance is an ethical and practical decision, not a purely technical one.

## 6. The ROC curve and AUC

### The idea
A classifier's 0.5 threshold is a choice, not a law. Lowering it catches more positives (higher recall) but raises false alarms. The **ROC curve** plots this trade-off across *all* thresholds: true-positive rate against false-positive rate. A model that hugs the top-left corner is excellent; the diagonal is random guessing.

The **AUC** (Area Under the Curve) summarises the whole curve in one number: 1.0 is perfect, 0.5 is random.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Probabilities for the positive class
y_proba = logreg.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

---
## ✏️ Exercises

### Exercise 1
Train a `KNeighborsClassifier` with `n_neighbors=1` and another with `n_neighbors=15` on the training data. Compare their test accuracy. Which is more likely to overfit, and why?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for k in [1, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(f'k={k}: test accuracy = {m.score(X_test, y_test):.3f}')
```

*k=1 classifies by the single nearest point, so it follows noise closely and tends to overfit. Larger k smooths the boundary and usually generalises better, another instance of the bias-variance trade-off.*
</details>

### Exercise 2
A disease classifier produces this confusion matrix (rows = actual, cols = predicted), with disease = positive:

```
                Pred Healthy   Pred Disease
Actual Healthy       90             10
Actual Disease        8             12
```
Compute **precision** and **recall** for the disease class by hand, then say which is more concerning here.

In [ ]:
# Your calculation here:


<details><summary>💡 Show solution</summary>

```python
TP = 12; FP = 10; FN = 8; TN = 90
precision = TP / (TP + FP)   # 12 / 22 = 0.545
recall    = TP / (TP + FN)   # 12 / 20 = 0.600
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
```

*Recall of 0.60 means the test misses 40% of real disease cases (8 false negatives), alarming for a medical screen, where missing a sick patient is the costliest error. Improving recall would be the priority.*
</details>

## 🔑 Key takeaways

- **Classification** predicts categories; logistic regression outputs probabilities via the **sigmoid**.
- **k-NN** votes among nearest neighbours; **decision trees** ask sequential threshold questions.
- Each algorithm produces a differently shaped **decision boundary**.
- The **confusion matrix** (TP, FP, FN, TN) underpins **precision**, **recall**, and **F1**.
- The **ROC curve / AUC** evaluate a classifier across all thresholds, vital when one error type is costlier than another.

---
**Next:** *Notebook 11: Beyond the Basics*: feature engineering, cross-validation, ensembles, and unsupervised learning, completing your classical-ML toolkit before deep learning.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*